# Training an AI for an Autonomous Intrusion Detection Agent

This notebook documents the end-to-end process of training a machine learning model for an autonomous Intrusion Detection System (IDS). The goal is to analyze network session data, compare several classification models, fine-tune the best performer to meet specific security goals, and finally, save the trained model for deployment in a live agent.

## 1. Setup and Imports

First, we import all the necessary libraries for data manipulation, modeling, evaluation, and saving.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
import joblib
import shap

## 2. Data Loading and Preprocessing

We load the dataset, handle missing values, and convert categorical text data into a numerical format that the models can understand using one-hot encoding.

In [ ]:
df = pd.read_csv("the_dataset.csv")

# Handle missing values in 'encryption_used'
df['encryption_used'] = df['encryption_used'].fillna('None')

# One-hot encode categorical features
df_processed = df.drop('session_id', axis=1)
categorical_cols = ['protocol_type', 'encryption_used', 'browser_type']
df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

# Convert boolean columns (True/False) to integers (1/0) for model compatibility
df_processed = df_processed.astype(int)

print("Data after preprocessing:")
df_processed.head()

## 3. Data Splitting

We split the data into a training set (for teaching the model) and a testing set (for evaluating its performance on unseen data).

In [ ]:
X = df_processed.drop('attack_detected', axis=1)
y = df_processed['attack_detected']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## 4. Model Iteration and Evaluation

We will now train and evaluate several models to find the best one for our task. For an IDS, our primary goal is to maximize **Recall** for the 'Attack' class (to minimize missed attacks), while keeping **Precision** reasonably high (to avoid too many false alarms).

### Model 1: Decision Tree (Baseline)

A simple Decision Tree is a good starting point. It's easy to understand but can be prone to overfitting.

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)

print("--- Decision Tree Performance ---")
print(confusion_matrix(y_test, dt_predictions))
print(classification_report(y_test, dt_predictions))

### Model 2: Random Forest

A Random Forest is an ensemble of many decision trees. It's more robust and less likely to overfit. We use `class_weight='balanced'` to tell the model to pay more attention to the minority 'Attack' class.

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print("--- Random Forest Performance (Default Threshold) ---")
print(confusion_matrix(y_test, rf_predictions))
print(classification_report(y_test, rf_predictions))

### Model 3: XGBoost (The Champion)

XGBoost is a powerful gradient boosting model known for its high performance. We use `scale_pos_weight` to handle the class imbalance, which is XGBoost's equivalent of `class_weight`.

In [ ]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(scale_pos_weight=ratio, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

print("--- XGBoost Performance (Default Threshold) ---")
print(confusion_matrix(y_test, xgb_predictions))
print(classification_report(y_test, xgb_predictions))

## 5. Fine-Tuning the Champion Model

The XGBoost model shows high precision but lower recall than desired. To fix this, we'll adjust its decision threshold. Instead of classifying an attack at a 50% probability, we'll lower the threshold to 30% to make the model more sensitive to potential threats.

In [ ]:
# Get the predicted probabilities from the XGBoost model
y_scores = xgb_model.predict_proba(X_test)[:, 1]

# Apply our tuned threshold
new_threshold = 0.3
tuned_predictions = (y_scores >= new_threshold).astype(int)

print(f"--- Final XGBoost Performance with Threshold = {new_threshold} ---")
print("\n--- Confusion Matrix (Tuned XGBoost) ---")
print(confusion_matrix(y_test, tuned_predictions))

print("\n--- Classification Report (Tuned XGBoost) ---")
print(classification_report(y_test, tuned_predictions))

## 6. Saving the Final Model

The tuned XGBoost model provides the best balance of performance for our needs. We will now save this trained model to a file using `joblib`. This file is the final 'brain' that our agent will use.

In [ ]:
MODEL_FILE = 'ids_agent_model.joblib'
joblib.dump(xgb_model, MODEL_FILE)

print(f"✅ Champion model successfully trained and saved to '{MODEL_FILE}'!")

## 7. Validation with XAI (SHAP)

As a final step, we'll load the saved model and initialize a SHAP explainer to confirm that our model is not only predictive but also interpretable.

In [ ]:
# Load the saved model
loaded_model = joblib.load(MODEL_FILE)

# Initialize the SHAP explainer with the training data as background
explainer = shap.TreeExplainer(loaded_model, X_train)

# Generate an explanation for the first sample in the test set
shap_values = explainer(X_test.iloc[[0]])
top_features = pd.Series(shap_values.values[0], index=X_test.columns).abs().nlargest(3)

explanation = "Threat flagged primarily due to:"
for feature, value in top_features.items():
    explanation += f"\n  - '{feature}' (Contribution score: {value:.2f})"

print("--- Example XAI Explanation ---")
print(explanation)

## Conclusion

The training process is complete. The fine-tuned XGBoost model has been validated and saved as `ids_agent_model.joblib`. This file is now ready to be deployed in our API-based agent.